In [1]:
# Import libraries
import kagglehub
import shutil
import os

import json
from datetime import datetime

In [7]:
# Download to default cache
path = kagglehub.dataset_download("Cornell-University/arxiv")

100%|██████████| 1.58G/1.58G [04:32<00:00, 6.25MB/s]

Extracting files...


In [8]:
# Move to your desired location
target_dir = "/Users/youhorng/Desktop/final-year-project/researchhub-ai-rag-system-final-year-project/backend/airflow/data"
os.makedirs(target_dir, exist_ok=True)
shutil.copytree(path, target_dir, dirs_exist_ok=True)

print("Data saved to:", target_dir)

Data saved to: /Users/youhorng/Desktop/final-year-project/researchhub-ai-rag-system-final-year-project/backend/airflow/data


In [2]:
# The path where you extracted the dataset in the previous cell
dataset_path = "/Users/youhorng/Desktop/final-year-project/researchhub-ai-rag-system-final-year-project/backend/airflow/data/arxiv-metadata-oai-snapshot.json"

In [ ]:
# ==========================================
# CONFIGURE YOUR PARAMETERS HERE:
# ==========================================
TARGET_YEARS = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]                      # e.g., Only get papers updated in 2023 or 2024
TARGET_CATEGORIES = ["cs.AI", "cs.LG", "stat.ML", "cs.DB", "cs.IR", 
                    "cs.DS", "cs.CL", "cs.CV", "eess.AS", "eess.IV", 
                    "cs.IT", "cs.PL", "cs.SE", "cs.DM", "cs.ET"] # Specifically AI and Machine Learning. 
                                                 # Note: Use ["cs.", "stat.ML"] if you want ALL computer science papers
# ==========================================
def is_relevant_category(paper_categories_str, target_categories):
    paper_cats = paper_categories_str.split(" ")
    for cat in paper_cats:
        for target in target_categories:
            # Check if category perfectly matches, or starts with target (e.g., target 'cs.' matches 'cs.AI')
            if cat.startswith(target) or cat == target:
                return True
    return False
def count_papers_by_year_and_category(file_path, target_years, target_categories):
    print(f"Counting papers for years {target_years} and categories {target_categories}...")
    total_processed = 0
    matched_count = 0
    
    with open(file_path, "r") as f:
        for line in f:
            total_processed += 1
            
            # Print progress every 500,000 scanned papers
            if total_processed % 500000 == 0:
                print(f"Scanned {total_processed:,} papers...")
                
            try:
                paper = json.loads(line)
                
                # 1. Filter by category
                if not is_relevant_category(paper.get("categories", ""), target_categories):
                    continue
                
                # 2. Filter by publication/update year
                update_date_str = paper.get("update_date", "")
                if update_date_str:
                    try:
                        # Extract the year from "YYYY-MM-DD" quickly
                        year = int(update_date_str.split("-")[0]) 
                        if year in target_years:
                            matched_count += 1
                    except ValueError:
                        pass
                        
            except json.JSONDecodeError:
                pass
                
    print("-" * 40)
    print(f"Total papers scanned in file: {total_processed:,}")
    print(f"Total papers matching your criteria: {matched_count:,}")
    return matched_count

# Run the counter
matched = count_papers_by_year_and_category(dataset_path, TARGET_YEARS, TARGET_CATEGORIES)
# Estimate limits (averaging 187 tokens per title + abstract based on our earlier calculation)
avg_tokens_per_paper = 187
estimated_tokens = matched * avg_tokens_per_paper
print(f"Estimated Jina embeddings tokens needed: {estimated_tokens:,.0f} tokens")
if estimated_tokens > 10000000:
    print("⚠️ WARNING: This exceeds your 10,000,000 free Jina token limit!")
else:
    print("✅ SUCCESS: This fits comfortably within your 10,000,000 free Jina token limit.")

Counting papers for years [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026] and categories ['cs.AI', 'cs.LG', 'stat.ML', 'cs.DB', 'cs.IR', 'cs.DS', 'cs.CL', 'cs.CV', 'eess.AS', 'eess.IV', 'cs.IT', 'cs.PL', 'cs.SE', 'cs.DM', 'cs.ET']...
Scanned 500,000 papers...
Scanned 1,000,000 papers...
Scanned 1,500,000 papers...
Scanned 2,000,000 papers...
Scanned 2,500,000 papers...
----------------------------------------
Total papers scanned in file: 2,975,294
Total papers matching your criteria: 662,014
Estimated Jina embeddings tokens needed: 123,796,618 tokens
⚠️ WARNING: This exceeds your 10,000,000 free Jina token limit!
